In [1]:
import os
import re
import glob
import tqdm

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# model_name = "Qwen/Qwen3-4B-Instruct-2507-FP8"
# model_name = "Qwen/Qwen2.5-7B-Instruct-1M"
# model_name = "Qwen/Qwen2.5-3B-Instruct"
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# load the tokenizer and the model
# print("Debug: Model load")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto",
    max_memory = {0:'13GiB'}
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [8]:
def genParQA(question,answer,N_gen=1):
    pqs = genParaphrasedQ(question, answer, N_gen)
    qas = []
    for q in pqs:
        qas.append(q+'|'+answer)
    return qas

def genParBatQA(questions,answers,N_gen=1):
    pqs = genParaphraseBatchdQ(questions, answers, N_gen)
    qas = []
    for iq in range(len(pqs)):
        qas.append(pqs[iq]+'|'+answers[iq])
    return qas


def genParaphraseBatchdQ(questions, answers, N_gen=1):
    system = "You are an expert language tool with extensive knowledge on surgical data. Rephrase the following question into natural, conversational language. Ask the same exact thing in a different way. Generate only in English, no other language. Do not change the underlying meaning, and ensure the original answer remains 100% accurate without needing extra elaboration. Do not say anything else than the paraphrased question.\n"
    
    texts = []
    for iq in range(len(questions)):
        prompt = f"Original question: {questions[iq]}\nOriginal answer: {answers[iq]}\nParaphrased Question:\n"
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": prompt}
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        texts.append(text)
        
    
    outputs = []
    for _ in range(N_gen):
        # print("Debug: Model Input")
        model_inputs = tokenizer(texts, padding=True, padding_side='left', return_tensors="pt").to(model.device)
        
        # print("Debug: Generate")
        # conduct text completion
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=16384
        ).to('cpu')

        model_inputs = model_inputs.to('cpu')

        # print(generated_ids)
        for igid in range(len(generated_ids)):
            output_ids = generated_ids[igid][len(model_inputs.input_ids[igid]):].tolist() 
            
            content = tokenizer.decode(output_ids, skip_special_tokens=True)
            # print("Original question:",question)
            # print("content:", content)
            content = content
            outputs.append(content)

        
        del model_inputs
        del output_ids
        del generated_ids
        
    return outputs

def genParaphrasedQ(question, answer, N_gen=1):
    system = "You are an expert language tool with extensive knowledge on surgical data. Rephrase the following question into natural, conversational language. Ask the same exact thing in a different way. Only give output in English, no other language. Do not change the underlying meaning, and ensure the original answer remains 100% accurate without needing extra elaboration. Do not say anything else than the paraphrased question.\n"
    prompt = f"Original question: {question}\nOriginal answer: {answer}\nParaphrased Question:\n"
    
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    
    outputs = []
    for _ in range(N_gen):
        # print("Debug: Model Input")
        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
        
        # print("Debug: Generate")
        # conduct text completion
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=16384
        )
        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
        
        content = tokenizer.decode(output_ids, skip_special_tokens=True)
        
        # print("Original question:",question)
        # print("content:", content)
        outputs.append(content)
    return outputs

In [9]:
genParaphrasedQ('What is the function currently being performed by the large needle driver during the surgical procedure?', 'grasp', N_gen=5)

['What role does the big needle driver play during the surgery?',
 'What role does the large needle driver play during the surgery?',
 'What role does the large needle driver play in the current surgical task?',
 'What role does the large needle driver play in the current surgical process?',
 'What task does this large needle driver perform during the surgery?']

In [10]:
genParQA('What is the function currently being performed by the large needle driver during the surgical procedure?', 'grasp', N_gen=5)

['What role does the large needle driver play in the current surgical process?|grasp',
 'What role does the big needle driver play in the current surgical process?|grasp',
 'What specific task does the large needle driver accomplish during the operation?|grasp',
 'What role does the large needle driver play during the surgery?|grasp',
 'What role does the large needle driver play during the surgery?|grasp']

In [11]:
qa = """Which anatomical structures are present?|abdominal_wall_cavity, liver, gut, omentum, gallbladder
Which tools are present?|grasper
What is the grasper doing?|grasp
Which tool is operating on gallbladder|grasper
What anatomy is at the top-left of the frame ?|abdominal_wall_cavity|[67.0, 85.5]
What anatomy is at the top-mid of the frame ?|liver|[212.0, 119.5]
What number of objects are both below the red top-left object and above the top-mid liver object?|0|single_and.json|count|NA
How many anatomys are both right of the abdominal_wall_cavity thing and to the left of the top-mid liver thing?|1|single_and.json|count|NA
How many things are both right of the abdominal_wall_cavity thing and on the left side of the top-mid liver object?|2|single_and.json|count|NA
What number of red anatomys are both to the right of the abdominal_wall_cavity object and left of the top-mid liver thing?|0|single_and.json|count|NA
What name is the thing that is within the silver top-mid thing and to the right of the white anatomy?|liver|single_and.json|query_component|NA
What is the name of the anatomy that is both to the right of the abdominal_wall_cavity thing and to the left of the yellow thing?|gallbladder|single_and.json|query_component|NA
What is the name of the anatomy that is both right of the red top-left abdominal_wall_cavity anatomy and to the left of the yellow object?|gallbladder|single_and.json|query_component|NA
There is a object that is below the grasper instrument and above the yellow thing?; what is its name?|abdominal_wall_cavity|single_and.json|query_component|NA
What is the color of the thing that is both below the top-left abdominal_wall_cavity anatomy and on the left side of the gut thing?|yellow|single_and.json|query_color|NA
There is a object that is both within the silver top-mid grasper instrument and within the gut object?; what is its color?|brown|single_and.json|query_color|NA
What is the color of the anatomy that is both within the silver top-mid grasper instrument and within the gut object?|brown|single_and.json|query_color|NA
There is a anatomy that is both below the top-left object and within the gut thing?; what is its color?|yellow|single_and.json|query_color|NA
What number of silver grasper things are there?|1|zero_hop.json|count|NA
How many brown bottom-left specimenbag anatomys are there?|0|zero_hop.json|count|NA
What number of brown anatomys are there?|1|zero_hop.json|count|NA
There is a liver thing; how many abdominal_wall_cavity anatomys are grasped by it?|0|one_hop.json|count|relate_actional
What number of gut anatomys are below the liver object?|1|one_hop.json|count|relate_spatial
There is a top-mid liver object; are there any bottom-right anatomys grasped by it?|False|one_hop.json|exist|relate_actional
There is a top-mid liver thing; are there any anatomys below it?|True|one_hop.json|exist|relate_spatial
The top-mid anatomy that is grasped by the top-mid grasper instrument is what color?|white|one_hop.json|query_color|relate_actional
There is a top-mid thing that is right of the top-mid grasper instrument; what color is it?|brown|one_hop.json|query_color|relate_spatial
There is a object on the right side of the liver anatomy; what type is it?|anatomy|one_hop.json|query_type|relate_spatial
There is a silver grasper thing that is to the left of the liver anatomy; what type is it?|instrument|one_hop.json|query_type|relate_spatial
There is a red object to the left of the silver top-mid grasper thing; what is it?|abdominal_wall_cavity|one_hop.json|query_component|relate_spatial
There is a top-mid anatomy that is right of the silver top-mid grasper object; what is it?|liver|one_hop.json|query_component|relate_spatial
"""
qal = qa.splitlines()
questions = []
answers = []
for qad in qal:
    qadl = qad.split('|')
    questions.append(qadl[0])
    answers.append('|'.join(qadl[1:]))

genParaphraseBatchdQ(questions,answers)
# genParBatQA(questions,answers)

['Which parts of the body are included?',
 'Which instruments are included?',
 'What action is the grasper performing?',
 'What surgical instrument is used for gallbladder manipulation?',
 'What part of the body is positioned at the upper left section?',
 'Which part of the body is located at the upper-middle section?',
 'How many items fall beneath the red top-left entity but exceed the top-middle liver?',
 'How many organs lie both to the right side of the abdominal cavity and to the left side of the top-middle part of the liver?',
 'How many entities are situated both to the lateral aspect of the abdominal cavity and anteriorly towards the inferior pole of the left lobe of the liver?',
 'How many red anatomical objects lie both to the right side of the abdominal wall cavity and to the left of the top-middle portion of the liver?',
 "What's the term for the object located beneath the silver upper part and situated to the immediate left of the white anatomical illustration?",
 'What\'

In [13]:
# SSG-VQA
out_flnms = []
in_flnms = []
in_flds = glob.glob('./SSGVQA/qa_txt/VID*')
for fldn in in_flds:
    in_fls = glob.glob(fldn+'/*.txt')
    for fln in in_fls:
        fll = fln.split('/')
        fll[1] += '_oot'
        flnn = '/'.join(fll)
        in_flnms.append(fln)
        out_flnms.append(flnn)
        os.makedirs(os.path.dirname(flnn), exist_ok=True)
# out_flnms

for i in tqdm.tqdm(range(len(in_flnms))):
    genqas = []
    with open(in_flnms[i],'r') as f:
        qas = f.read().splitlines()

    # Batched
    questions = []
    answers = []
    for qad in qas:
        qadl = qad.split('|')
        questions.append(qadl[0])
        answers.append('|'.join(qadl[1:]))
    
    genqa = genParBatQA(questions,answers)
    for gqa in genqa:
        genqas.append(gqa)

    # # Individual
    # for qa in qas:
    #     qal = qa.split('|')
    #     question = qal[0]
    #     answer = '|'.join(qal[1:])
    #     # print(question,answer)
    #     genqa = genParQA(question,answer)
    #     for gqa in genqa:
    #         genqas.append(gqa)

            
    with open(out_flnms[i],'w') as f:
        qas = f.writelines(f"{line}\n" for line in genqas)

  0%|          | 0/24252 [00:00<?, ?it/s]

  1%|          | 151/24252 [01:10<3:06:46,  2.15it/s]


KeyboardInterrupt: 

In [14]:
# SSG-VQA Temporal
# SSG-VQA
out_flnms = []
in_flnms = []
fldn = './ssgqa_qa_temporal/'
in_fls = glob.glob(fldn+'/*.txt')
for fln in in_fls:
    fll = fln.split('/')
    fll[1] += '_oot'
    flnn = '/'.join(fll)
    in_flnms.append(fln)
    out_flnms.append(flnn)
    os.makedirs(os.path.dirname(flnn), exist_ok=True)
# out_flnms

for i in tqdm.tqdm(range(len(in_flnms))):
    genqas = []
    with open(in_flnms[i],'r') as f:
        qas = f.read().splitlines()

    # Batched
    questions = []
    answers = []
    for qad in qas:
        qadl = qad.split('|')
        questions.append(qadl[0])
        answers.append('|'.join(qadl[1:]))
    
    genqa = genParBatQA(questions,answers)
    for gqa in genqa:
        genqas.append(gqa)

    # # Individual
    # for qa in qas:
    #     qal = qa.split('|')
    #     question = qal[0]
    #     answer = '|'.join(qal[1:])
    #     # print(question,answer)
    #     genqa = genParQA(question,answer)
    #     for gqa in genqa:
    #         genqas.append(gqa)

            
    with open(out_flnms[i],'w') as f:
        qas = f.writelines(f"{line}\n" for line in genqas)

100%|██████████| 50/50 [01:55<00:00,  2.31s/it]
